<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 08 - Evaluación RA1: Práctica

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Contenido:** Ejercicio de evaluación - Análisis de supermercados

---

## Caso: Análisis de datos de una cadena de supermercados

Este notebook simula el ejercicio de evaluación del RA1.

## 1. Generación de datos de ejemplo

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

# Generar datos de ventas
n_ventas = 1000
fechas = pd.date_range('2024-01-01', periods=90, freq='D')

ventas = pd.DataFrame({
    'venta_id': range(1, n_ventas + 1),
    'fecha': np.random.choice(fechas, n_ventas),
    'producto_id': np.random.randint(1, 51, n_ventas),
    'tienda_id': np.random.randint(1, 6, n_ventas),
    'cantidad': np.random.randint(1, 10, n_ventas),
    'precio_unitario': np.round(np.random.uniform(0.5, 50, n_ventas), 2)
})

# Añadir algunos nulos y duplicados para análisis de calidad
ventas.loc[np.random.choice(ventas.index, 20), 'cantidad'] = np.nan
ventas = pd.concat([ventas, ventas.sample(15)], ignore_index=True)  # Duplicados

# Productos
productos = pd.DataFrame({
    'producto_id': range(1, 51),
    'nombre': [f'Producto_{i}' for i in range(1, 51)],
    'categoria': np.random.choice(['Alimentación', 'Bebidas', 'Limpieza', 'Hogar'], 50)
})

# Tiendas
tiendas = pd.DataFrame({
    'tienda_id': range(1, 6),
    'nombre': ['Málaga Centro', 'Málaga Este', 'Marbella', 'Fuengirola', 'Torremolinos'],
    'region': ['Centro', 'Este', 'Costa', 'Costa', 'Costa']
})

print("=== DATASETS CARGADOS ===")
print(f"Ventas: {len(ventas)} registros")
print(f"Productos: {len(productos)} registros")
print(f"Tiendas: {len(tiendas)} registros")

## 2. Análisis de calidad

In [ ]:
print("=== ANÁLISIS DE CALIDAD ===")

# Completitud
print("\n📊 COMPLETITUD:")
for col in ventas.columns:
    nulos = ventas[col].isna().sum()
    pct = (1 - nulos/len(ventas)) * 100
    estado = "✅" if pct == 100 else "⚠️"
    print(f"  {estado} {col}: {pct:.1f}% completo ({nulos} nulos)")

# Duplicados
print("\n🔄 DUPLICADOS:")
duplicados = ventas.duplicated().sum()
print(f"  Total duplicados: {duplicados}")
print(f"  Porcentaje: {duplicados/len(ventas)*100:.2f}%")

# Valores anómalos
print("\n⚠️ VALORES ANÓMALOS:")
precios_negativos = (ventas['precio_unitario'] < 0).sum()
cantidades_cero = (ventas['cantidad'] == 0).sum()
print(f"  Precios negativos: {precios_negativos}")
print(f"  Cantidades cero: {cantidades_cero}")

## 3. Limpieza y transformación

In [ ]:
print("=== LIMPIEZA DE DATOS ===")

# Eliminar duplicados
ventas_clean = ventas.drop_duplicates()
print(f"Eliminados {len(ventas) - len(ventas_clean)} duplicados")

# Rellenar nulos en cantidad con la mediana
mediana_cantidad = ventas_clean['cantidad'].median()
ventas_clean['cantidad'] = ventas_clean['cantidad'].fillna(mediana_cantidad)
print(f"Nulos en cantidad rellenados con mediana: {mediana_cantidad}")

# Calcular total
ventas_clean['total'] = ventas_clean['cantidad'] * ventas_clean['precio_unitario']

print(f"\nDataset limpio: {len(ventas_clean)} registros")

## 4. Combinar datasets (Modelo dimensional)

In [ ]:
# Unir ventas con productos y tiendas
df = ventas_clean.merge(productos, on='producto_id')
df = df.merge(tiendas, on='tienda_id')

print("=== MODELO DIMENSIONAL ===")
print(f"\nTabla de hechos (ventas) combinada con dimensiones")
print(f"Total registros: {len(df)}")
print(f"\nColumnas: {df.columns.tolist()}")
print(f"\nMuestra:")
df[['venta_id', 'fecha', 'nombre_x', 'categoria', 'nombre_y', 'total']].head()

## 5. Análisis de negocio

In [ ]:
import plotly.express as px

print("=== ANÁLISIS DE NEGOCIO ===")

# Top 5 productos más vendidos
print("\n📦 TOP 5 PRODUCTOS MÁS VENDIDOS:")
top_productos = df.groupby('nombre_x')['cantidad'].sum().sort_values(ascending=False).head(5)
print(top_productos)

# Ventas por tienda
print("\n🏪 VENTAS POR TIENDA:")
ventas_tienda = df.groupby('nombre_y').agg({
    'total': 'sum',
    'venta_id': 'count'
}).rename(columns={'venta_id': 'num_transacciones'})
ventas_tienda['ticket_medio'] = ventas_tienda['total'] / ventas_tienda['num_transacciones']
print(ventas_tienda.round(2))

# Visualización
fig = px.bar(ventas_tienda.reset_index(), x='nombre_y', y='total',
             title='Ventas por Tienda',
             labels={'nombre_y': 'Tienda', 'total': 'Total Ventas (€)'})
fig.show()

In [ ]:
# Ventas por categoría
print("\n📊 VENTAS POR CATEGORÍA:")
ventas_categoria = df.groupby('categoria')['total'].sum().sort_values(ascending=False)
print(ventas_categoria)

fig = px.pie(values=ventas_categoria.values, names=ventas_categoria.index,
             title='Distribución de Ventas por Categoría')
fig.show()

## 6. Reflexión sobre complejidad

**¿Qué complejidad tienen las operaciones realizadas?**

| Operación | Complejidad | Escalable a 100M? |
|-----------|-------------|-------------------|
| `drop_duplicates()` | O(n) | ⚠️ Requiere memoria |
| `groupby().sum()` | O(n) | ✅ Sí |
| `merge()` (hash join) | O(n + m) | ✅ Sí |
| `sort_values()` | O(n log n) | ✅ Sí |

**¿Cómo escalar a 100M de registros?**
- Usar PySpark para procesamiento distribuido
- Particionar datos por fecha/tienda
- Almacenar en formato Parquet

---

## Resumen de criterios evaluados (RA1)

- ✅ CE a) Conceptos de matemática discreta y complejidad
- ✅ CE b) Extracción automática de información
- ✅ CE c) Combinación de diferentes fuentes
- ✅ CE d) Modelado de datos
- ✅ CE e) Calidad de datos